In [1]:
from pathlib import Path
import sys

# common/ lives in BuildingWithClaudeAPI/, a sibling of this folder.
# Walk up from the current directory until BuildingWithClaudeAPI/common is found.
sys.path.append(r"C:\Users\SValathalaKu\Data\claude-bedrock-learning\BuildingWithClaudeAPI")
from common.aws_config import aws_env_setup
#from common.bedrock_config import get_anthropic_client

aws_env_setup()
#client, model = get_anthropic_client()


In [2]:
from anthropic import AnthropicBedrock

def get_anthropic_client():

     client = AnthropicBedrock(
      aws_region="ap-southeast-2"
      )
     model = "au.anthropic.claude-haiku-4-5-20251001-v1:0"
     return client, model
client, model = get_anthropic_client()

In [3]:
from pathlib import Path
from pypdf import PdfReader

def read_pdf_file(file_name: str) -> str:
    """Return the extracted text of a PDF located in the same directory as this notebook."""
    if not file_name.lower().endswith(".pdf"):
        file_name += ".pdf"

    pdf_path = Path.cwd() / file_name
    if not pdf_path.is_file():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    reader = PdfReader(pdf_path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def read_text_file(file_name: str) -> str:
    """Return the contents of a text file in the same directory as this notebook, read as UTF-8."""
    text_path = Path.cwd() / file_name
    if not text_path.is_file():
        raise FileNotFoundError(f"File not found: {text_path}")

    with open(text_path, "r", encoding="utf-8") as f:
        return f.read()

In [4]:
# --------------------------------------------------
# Ring 1: Tool implementations
# --------------------------------------------------

def read_text_file(filename: str) -> str:
    path = Path(filename)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {filename}")

    return path.read_text(encoding="utf-8")


def count_file_words(filename: str) -> dict:
    content = read_text_file(filename)

    return {
        "filename": filename,
        "word_count": len(content.split())
    }


def search_text_file(filename: str, search_term: str) -> dict:
    content = read_text_file(filename)
    matching_lines = []

    for line_number, line in enumerate(content.splitlines(), start=1):
        if search_term.lower() in line.lower():
            matching_lines.append({
                "line_number": line_number,
                "text": line
            })

    return {
        "filename": filename,
        "search_term": search_term,
        "matches": matching_lines
    }

In [5]:
# --------------------------------------------------
# Tool schemas shown to Claude
# --------------------------------------------------

tools = [
    {
        "name": "read_text_file",
        "description": (
            "Read the complete contents of a UTF-8 text file. "
            "Use when the user wants to understand or summarise the file."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {"type": "string"}
            },
            "required": ["filename"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "name": "count_file_words",
        "description": (
            "Count the words in a text file. "
            "Use only when the user asks for the file's word count."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {"type": "string"}
            },
            "required": ["filename"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "name": "search_text_file",
        "description": (
            "Search a text file for a specific word or phrase and return "
            "matching lines with line numbers. Do not use this tool when "
            "the complete file contents are required."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {"type": "string"},
                "search_term": {"type": "string"}
            },
            "required": ["filename", "search_term"],
            "additionalProperties": False
        },
        "strict": True
    }
]

In [6]:
# --------------------------------------------------
# Tool dispatcher
# --------------------------------------------------

tool_functions = {
    "read_text_file": read_text_file,
    "count_file_words": count_file_words,
    "search_text_file": search_text_file
}


def run_tool(tool_name: str, tool_input: dict):
    function = tool_functions.get(tool_name)

    if function is None:
        raise ValueError(f"Unknown tool: {tool_name}")

    return function(**tool_input)

In [7]:
import json 
# --------------------------------------------------
# Ring 2: Agentic loop
# --------------------------------------------------

def run_agent(user_question: str, max_turns: int = 6):
    messages = [
        {
            "role": "user",
            "content": user_question
        }
    ]

    for turn in range(max_turns):
        response = client.messages.create(
            model=model,
            max_tokens=500,
            tools=tools,
            tool_choice={
                "type": "auto",
                "disable_parallel_tool_use": False # this will be False for parallel run 
                # The applicaiton should use asyc to run multiple functions in parallel . not the job of Claude 
            },
            messages=messages
        )

        print(f"Turn {turn + 1}: {response.stop_reason}")

        # Final response: exit the agentic loop.
        if response.stop_reason == "end_turn":
            return "\n".join(
                block.text
                for block in response.content
                if block.type == "text"
            )

        if response.stop_reason != "tool_use":
            raise RuntimeError(
                f"Unexpected stop reason: {response.stop_reason}"
            )

        # Preserve Claude's tool request in conversation history.
        messages.append({
            "role": "assistant",
            "content": response.content
        })

        tool_results = []

        # Parallel use is disabled, but this still safely handles
        # every tool_use block returned in the response.
        for block in response.content:
            if block.type != "tool_use":
                continue

            try:
                output = run_tool(block.name, block.input)

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(output)
                })

            except FileNotFoundError as error:
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps({
                        "errorCategory": "validation",
                        "isRetryable": False,
                        "description": str(error)
                    }),
                    "is_error": True
                })

            except Exception as error:
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps({
                        "errorCategory": "transient",
                        "isRetryable": True,
                        "description": str(error)
                    }),
                    "is_error": True
                })

        # Return tool results to Claude.
        messages.append({
            "role": "user",
            "content": tool_results
        })

    return "Agent stopped because the maximum turn limit was reached."


result = run_agent(
    """
    Read data3.txt, tell me its word count,
    and find every line that mentions 'tool'.
    Then provide a concise summary.
    """
)

print(result)

Turn 1: tool_use
Turn 2: end_turn
## Summary

**Word Count:** 122 words

**Lines Mentioning 'tool':** 12 matches found

| Line # | Text |
|--------|------|
| 1 | Yes. run_tool() can call any function registered in the tool_functions dictionary. |
| 4 | tool_functions = { |
| 12 | tool_name comes from Claude's tool_use block. |
| 13 | .get(tool_name) retrieves the matching Python function. |
| 14 | function(**tool_input) executes that function using Claude's input as named arguments. |
| 19 | tool_name = "search_text_file" |
| 21 | tool_input = { |
| 34 | The function must be registered in tool_functions. |
| 35 | The dictionary key must match the tool schema's name. |
| 36 | The keys in tool_input must match the function's parameter names. |
| 37 | The function shown is synchronous. An async function requires await function(**tool_input). |
| 39 | Memory: Tool schema tells Claude what exists; dispatcher maps the name; Python function performs the work. |

**Concise Summary:**
The file 

PARALLEL CALL

In [9]:
import asyncio
import json

async def run_agent(user_question: str, max_turns: int = 6):
    messages = [{"role": "user", "content": user_question}]

    for turn in range(max_turns):
        response = client.messages.create(
            model=model,
            max_tokens=500,
            tools=tools,
            tool_choice={"type": "auto", "disable_parallel_tool_use": False},
            messages=messages,
        )

        print(f"Turn {turn + 1}: {response.stop_reason}")

        if response.stop_reason == "end_turn":
            return "\n".join(
                block.text for block in response.content if block.type == "text"
            )

        if response.stop_reason != "tool_use":
            raise RuntimeError(f"Unexpected stop reason: {response.stop_reason}")

        messages.append({"role": "assistant", "content": response.content})

        tool_blocks = [b for b in response.content if b.type == "tool_use"]

        async def execute(block):
            try:
                # run_tool is blocking, so move it off the event loop
                output = await asyncio.to_thread(run_tool, block.name, block.input)
                return {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(output),
                }
            except FileNotFoundError as error:
                return {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps({
                        "errorCategory": "validation",
                        "isRetryable": False,
                        "description": str(error),
                    }),
                    "is_error": True,
                }
            except Exception as error:
                return {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps({
                        "errorCategory": "transient",
                        "isRetryable": True,
                        "description": str(error),
                    }),
                    "is_error": True,
                }

        # all requested tools run at the same time; results keep block order
        tool_results = await asyncio.gather(*(execute(b) for b in tool_blocks))

        messages.append({"role": "user", "content": list(tool_results)})

    return "Agent stopped because the maximum turn limit was reached."


result = await run_agent(
    """
    Read sajin.txt, tell me its word count,
    and find every line that mentions 'tool'.
    Then provide a concise summary.
    """
)
print(result)

Turn 1: tool_use
Turn 2: end_turn
## Summary

**Word Count:** 629 words

**Lines Mentioning 'tool':** The search found 32 lines containing the word 'tool' (shown above).

**Concise Summary:**

The file is documentation about "Strict Tool Use" in Claude's API. It explains how to enforce JSON Schema compliance when Claude calls functions by enabling strict mode (setting `strict: true`). Key points include:

- **Purpose:** Guarantees that Claude provides correctly typed arguments and all required fields, preventing runtime errors and incompatible data types (e.g., "2" vs 2)
- **Benefits:** Type-safe parameters, no need for validation/retries, and production-ready agents at scale
- **Implementation:** Define a JSON Schema for the tool's input, add `strict: true` to the tool definition, and handle the resulting tool calls
- **Use Cases:** Validated tool inputs, agentic workflows with multiple tools, and complex nested properties
- **Data Handling:** Tool schemas are cached for 24 hours, and